# Learning Urban Crimes Representation

## Packages

In [ ]:
# Packages needed
import pandas as pd # for data manipulation
import numpy as np #for numeric calculations and making simulated data.
import seaborn as sns #stylized data visualization
import matplotlib.pyplot as plt #basic data visualization
from matplotlib.colors import Normalize

# Other packages
import re
import warnings
warnings.filterwarnings('ignore')

# Maps packages
import geopandas as gpd
from shapely.geometry import Point, box
from mpl_toolkits.axes_grid1 import make_axes_locatable
import folium
from folium.plugins import HeatMap

# summary export packages
import json
from pathlib import Path

## Exploratory Data Analysis

### Configuration

In [ ]:
# pandas display options
pd.set_option("display.max_columns", 50) # to show all columns when printing a dataframe (limited to 50 cols)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") # to show floats with 2 decimals and commas as thousand separators

### Reading data

In [ ]:
# auxiliary maps

# original csv data types
dtype_map = {
    "anio_inicio": "Int64",
    "mes_inicio": "string",
    "fecha_inicio": "string",
    "hora_inicio": "string",
    "anio_hecho": "Float64",
    "mes_hecho": "string",
    "fecha_hecho": "string",
    "hora_hecho": "string",
    "delito": "string",
    "categoria_delito": "string",
    "competencia": "string",
    "fiscalia": "string",
    "agencia": "string",
    "unidad_investigacion": "string",
    "colonia_hecho": "string",
    "colonia_catalogo": "string",
    "alcaldia_hecho": "string",
    "alcaldia_catalogo": "string",
    "municipio_hecho": "string",
    "latitud": "Float64",
    "longitud": "Float64",
}

# day names from english to spanish
dias_map = {
    "Monday": "LUNES",
    "Tuesday": "MARTES",
    "Wednesday": "MIÉRCOLES",
    "Thursday": "JUEVES",
    "Friday": "VIERNES",
    "Saturday": "SÁBADO",
    "Sunday": "DOMINGO"
}


# months from english to spanish
meses_map = {
    "january": "Enero", "february": "Febrero", "march": "Marzo",
    "april": "Abril", "may": "Mayo", "june": "Junio",
    "july": "Julio", "august": "Agosto", "september": "Septiembre",
    "october": "Octubre", "november": "Noviembre", "december": "Diciembre"
}

In [ ]:

# access data with proper data types
data_path = "..\data\carpetasFGJ_acumulado_2025_01.csv"
data = pd.read_csv(data_path, dtype=dtype_map)

#### Initial exploration

In [ ]:
# print data info and first few rows to check registers
data

,anio_inicio,mes_inicio,fecha_inicio,hora_inicio,anio_hecho,mes_hecho,fecha_hecho,hora_hecho,delito,categoria_delito,competencia,fiscalia,agencia,unidad_investigacion,colonia_hecho,colonia_catalogo,alcaldia_hecho,alcaldia_catalogo,municipio_hecho,latitud,longitud
0,2016,Enero,2016-01-01,00:00:00,"2,015.00",Diciembre,2015-12-31,16:30:00,LESIONES CULPOSAS POR TRANSITO VEHICULAR EN CO...,DELITO DE BAJO IMPACTO,<NA>,INVESTIGACIÓN EN TLALPAN,TLP-4,UI-2CD,JARDINES EN LA MONTAÑA,Jardines En La Montaña,TLALPAN,<NA>,CDMX,19.30,-99.21
1,2016,Enero,2016-01-01,00:00:00,"2,015.00",Diciembre,2015-12-31,22:40:00,ROBO A PASAJERO A BORDO DE TAXI CON VIOLENCIA,ROBO A PASAJERO A BORDO DE TAXI CON VIOLENCIA,<NA>,INVESTIGACIÓN EN TLALPAN,TLP-1,UI-2CD,LOMAS DE PADIERNA,Lomas De Padierna,TLALPAN,<NA>,CDMX,19.29,-99.22
2,2016,Enero,2016-01-01,00:00:00,"2,016.00",Enero,2016-01-01,00:20:00,ROBO A TRANSEUNTE EN VIA PUBLICA CON VIOLENCIA,ROBO A TRANSEUNTE EN VÍA PÚBLICA CON Y SIN VIO...,<NA>,INVESTIGACIÓN EN IZTAPALAPA,IZP-2,UI-2CD,SAN ANTONIO CULHUACÁN,Barrio San Antonio Culhuacan,IZTAPALAPA,<NA>,CDMX,19.34,-99.11
3,2016,Enero,2016-01-01,00:00:00,"2,015.00",Diciembre,2015-12-31,22:00:00,ROBO DE VEHICULO DE SERVICIO PARTICULAR SIN VI...,ROBO DE VEHÍCULO CON Y SIN VIOLENCIA,<NA>,INVESTIGACIÓN EN GUSTAVO A. MADERO,GAM-8,UI-2CD,SAN JUAN DE ARAGÓN II SECCIÓN,San Juan De Aragon Ii Seccion,GUSTAVO A. MADERO,<NA>,CDMX,19.45,-99.09
4,2016,Enero,2016-01-01,00:00:00,"2,015.00",Diciembre,2015-12-31,22:30:00,HOMICIDIOS INTENCIONALES (OTROS),HOMICIDIO DOLOSO,<NA>,INVESTIGACIÓN EN BENITO JUÁREZ,BJ-1,UI-2SD,NATIVITAS,Nativitas,BENITO JUAREZ,<NA>,CDMX,19.38,-99.14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2098738,2024,Noviembre,2024-11-25 19:35:00,19:35:00,"2,024.00",Noviembre,2024-11-25 17:32:00,17:32:00,TENTATIVA DE EXTORSION,DELITO DE BAJO IMPACTO,FUERO COMUN,FISCALÍA DE INVESTIGACIÓN DEL DELITO DE SECUESTRO,E,2 CON DETENIDO 2 C/D,MIGUEL HIDALGO 3A SECCIÓN,Miguel Hidalgo 3a Seccion,TLALPAN,<NA>,CDMX,19.28,-99.21
2098739,2024,Noviembre,2024-11-26 19:53:00,19:53:00,"2,024.00",Noviembre,2024-11-25 11:20:00,11:20:00,EXTORSION,DELITO DE BAJO IMPACTO,FUERO COMUN,FISCALÍA DE INVESTIGACIÓN TERRITORIAL EN IZTAP...,IZP-4,UI-3SD,SANTA BÁRBARA,Barrio Santa Barbara,IZTAPALAPA,<NA>,CDMX,19.36,-99.10
2098740,2024,Noviembre,2024-11-28 16:27:00,16:27:00,"2,024.00",Noviembre,2024-11-28 13:15:00,13:15:00,EXTORSION,DELITO DE BAJO IMPACTO,FUERO COMUN,FISCALÍA DE INVESTIGACIÓN DEL DELITO DE SECUESTRO,E,2 CON DETENIDO 2 C/D,AMPLIACIÓN TORRE BLANCA,Pensil Norte,MIGUEL HIDALGO,<NA>,CDMX,19.45,-99.20
2098741,2024,Noviembre,2024-11-28 23:42:00,23:42:00,"2,024.00",Noviembre,2024-11-26 02:50:00,02:50:00,EXTORSION,DELITO DE BAJO IMPACTO,FUERO COMUN,FISCALÍA DE INVESTIGACIÓN DEL DELITO DE SECUESTRO,E,2 CON DETENIDO 2 C/D,ESPERANZA,Esperanza,CUAUHTEMOC,<NA>,CDMX,19.42,-99.13
